In [66]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [105]:
schedules_polars = nfl.load_schedules(seasons=True)
schedules = schedules_polars.to_pandas()
schedules = schedules.dropna(subset=["spread_line", "result"])
schedules = schedules[schedules['game_type'] == 'REG'] #Filter for regular season games
schedules.to_csv('schedules.csv', index=False)

In [106]:
#Split into train and test sets

#TODO: FIGURE OUT HOW TO CONVERT ANY KEY COLUMNS I WANT INTO NUM, STRING NOT ALLOWED FOR SCALER AND IN GENERAL
X = schedules[['spread_line', 'week']]
y = schedules['result'] > 0 #Convert result from point dif to binary outcome 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Scale features (put everything on scale 0-1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [107]:
model = LogisticRegression(max_iter=10000)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=10000)

In [108]:
#Do prediction on test set
y_pred = model.predict(X_test_scaled)

In [109]:
# Test model accuracy results
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Loss", "Win"]))

Model Accuracy: 0.6679

Classification Report:
              precision    recall  f1-score   support

        Loss       0.63      0.52      0.57       590
         Win       0.69      0.78      0.73       804

    accuracy                           0.67      1394
   macro avg       0.66      0.65      0.65      1394
weighted avg       0.66      0.67      0.66      1394



In [77]:
week1 =schedules[(schedules['week'] ==1) & (schedules['season'] == 2026)]
week1 = week1.reset_index(drop=True)
week1_scaled = scaler.transform(week1[['spread_line', 'week']])
week1_probs = model.predict_proba(week1_scaled)[:, 1]
print(week1_probs)


[0.59007215 0.60718269 0.37980078 0.60718269 0.71789325 0.43125576
 0.36317378 0.75909051 0.60718269 0.53757881 0.78417133 0.59007215
 0.53757881 0.67268869 0.37980078 0.57274074]


In [78]:
for index, row in week1.iterrows():
    print(f"Game: {row['away_team']} @ {row['home_team']}, Predicted Win Probability for Home Team: {week1_probs[index]:.4f}")

Game: NE @ SEA, Predicted Win Probability for Home Team: 0.5901
Game: SF @ LA, Predicted Win Probability for Home Team: 0.6072
Game: CHI @ CAR, Predicted Win Probability for Home Team: 0.3798
Game: TB @ CIN, Predicted Win Probability for Home Team: 0.6072
Game: NO @ DET, Predicted Win Probability for Home Team: 0.7179
Game: BUF @ HOU, Predicted Win Probability for Home Team: 0.4313
Game: BAL @ IND, Predicted Win Probability for Home Team: 0.3632
Game: CLE @ JAX, Predicted Win Probability for Home Team: 0.7591
Game: ATL @ PIT, Predicted Win Probability for Home Team: 0.6072
Game: NYJ @ TEN, Predicted Win Probability for Home Team: 0.5376
Game: ARI @ LAC, Predicted Win Probability for Home Team: 0.7842
Game: MIA @ LV, Predicted Win Probability for Home Team: 0.5901
Game: GB @ MIN, Predicted Win Probability for Home Team: 0.5376
Game: WAS @ PHI, Predicted Win Probability for Home Team: 0.6727
Game: DAL @ NYG, Predicted Win Probability for Home Team: 0.3798
Game: DEN @ KC, Predicted Win Pr

In [93]:
#Account for points based on spread. 1pt for favorite, 2 point for underdog. 3 points for 7 pt underdog
#add cols for homeIsFavorite (bool), 7ptUnderdog (bool), evHome (float), evAway (float), bestEV ("Home or Away")
week1_results = pd.DataFrame()
#Favorites
week1_results.insert(0, 'favorite', np.where(
    week1['spread_line'] >= 0, 
    week1['home_team'], 
    week1['away_team'])
)
#Underdogs
week1_results['underdog'] = np.where(
    week1['spread_line'] >= 0, 
    week1['away_team'], 
    week1['home_team'])
#Spread
week1_results['spread'] = np.where(
    week1['home_team'] == week1_results['favorite'],
    week1['spread_line'],
    week1['spread_line'] * -1,
)
#Favorite Win Prob
week1_results['home_win_prob'] = np.where(
    week1['home_team'] == week1_results['favorite'],
    week1_probs,
    1 - week1_probs,
)
#Underdog Win Prob
week1_results['underdog_win_prob'] = 1 - week1_results['home_win_prob']
#EV Home
week1_results['favorite_ev'] = week1_results['home_win_prob'] * 1
#Is Underdog 3pt (7+ point spread)
week1_results['is_3pt_underdog'] = week1_results['spread'] >= 7
#Ev Away
week1_results['underdog_ev'] = np.where(
    week1_results['is_3pt_underdog'],
    week1_results['underdog_win_prob'] * 3,
    week1_results['underdog_win_prob'] * 2
)
#Team Pick (highest EV)
week1_results['pick'] = np.where(
    week1_results['favorite_ev'] >= week1_results['underdog_ev'],
    week1_results['favorite'],
    week1_results['underdog']
)
print(week1_results)

   favorite underdog  spread  home_win_prob  underdog_win_prob  favorite_ev  \
0       SEA       NE     3.0       0.590072           0.409928     0.590072   
1        LA       SF     3.5       0.607183           0.392817     0.607183   
2       CHI      CAR     3.0       0.620199           0.379801     0.620199   
3       CIN       TB     3.5       0.607183           0.392817     0.607183   
4       DET       NO     7.0       0.717893           0.282107     0.717893   
5       BUF      HOU     1.5       0.568744           0.431256     0.568744   
6       BAL      IND     3.5       0.636826           0.363174     0.636826   
7       JAX      CLE     8.5       0.759091           0.240909     0.759091   
8       PIT      ATL     3.5       0.607183           0.392817     0.607183   
9       TEN      NYJ     1.5       0.537579           0.462421     0.537579   
10      LAC      ARI     9.5       0.784171           0.215829     0.784171   
11       LV      MIA     3.0       0.590072         